In [1]:
# Setup imports
from nntool.api import NNGraph
from nntool.api.utils import model_settings, quantization_options
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
#%matplotlib widget
import logging
logging.basicConfig(level=logging.INFO)

In [2]:
# Load the graph and visualize it
G = NNGraph.load_graph("patchifier_rpg.onnx")
max_activ_size, total_params = G.total_memory_usage
ops = G.total_ops
#G.draw(filepath="draw", view=True)

# Quantization and fusion
G.adjust_order()
G.fusions('scaled_match_group')
G.fusions('expression_matcher')

INFO:nntool.graph.manipulations.adjust_order:adding transposes to correct tensor order for AT kernels
INFO:nntool.graph.manipulations.eliminate_transposes.eliminate_transposes:no transposes to eliminate found
INFO:nntool.graph.nngraph:update graph dimensions
INFO:nntool.graph.nngraph:update graph dimensions
INFO:nntool.graph.manipulations.eliminate_transposes.eliminate_transposes:no further transpose sequences found
INFO:nntool.graph.nngraph:adjusted order
INFO:nntool.graph.matches.matchers.duplicate_operations_out:removed duplicates from _matching_feat_encoder_conv1_Conv_reshape_in _ctx_feat_encoder_conv1_Conv_reshape_in
INFO:nntool.graph.matches.matchers.duplicate_operations_out:removed duplicates from _matching_feat_encoder_conv1_Conv_reshape_in _scorer_scorer_scorer_0_Conv_reshape_in
INFO:nntool.graph.matches.matcher:++ fusion duplicate_operations_out modified graph
INFO:nntool.graph.matches.matchers.remove_noops:removing _matching_feat_encoder_conv1_Conv_reshape_out that does noth

get_output_size: _matching_feat_encoder_conv1_Conv has a padding setting not to SAME but in reality it is, changing the original padding from (3, 3)x(3, 3) to (3, 2)x(3, 2)
get_output_size: _matching_feat_encoder_layer2_layer2_0_conv1_Conv has a padding setting not to SAME but in reality it is, changing the original padding from (1, 1)x(1, 1) to (1, 0)x(1, 0)
get_output_size: _ctx_feat_encoder_conv1_Conv has a padding setting not to SAME but in reality it is, changing the original padding from (3, 3)x(3, 3) to (3, 2)x(3, 2)
get_output_size: _ctx_feat_encoder_layer2_layer2_0_conv1_Conv has a padding setting not to SAME but in reality it is, changing the original padding from (1, 1)x(1, 1) to (1, 0)x(1, 0)


INFO:nntool.graph.matches.matchers.fuse_gap_convs:fusing nodes _matching_feat_encoder_conv1_Conv,_matching_feat_encoder_relu1_Relu into _matching_feat_encoder_conv1_Conv_fusion
INFO:nntool.graph.matches.matchers.fuse_gap_convs:fusing nodes _matching_feat_encoder_layer1_layer1_0_conv1_Conv,_matching_feat_encoder_layer1_layer1_0_relu_Relu into _matching_feat_encoder_layer1_layer1_0_conv1_Conv_fusion
INFO:nntool.graph.matches.matchers.fuse_gap_convs:fusing nodes _matching_feat_encoder_layer1_layer1_0_conv2_Conv,_matching_feat_encoder_layer1_layer1_0_relu_1_Relu into _matching_feat_encoder_layer1_layer1_0_conv2_Conv_fusion
INFO:nntool.graph.matches.matchers.fuse_gap_convs:fusing nodes _matching_feat_encoder_layer1_layer1_1_conv1_Conv,_matching_feat_encoder_layer1_layer1_1_relu_Relu into _matching_feat_encoder_layer1_layer1_1_conv1_Conv_fusion
INFO:nntool.graph.matches.matchers.fuse_gap_convs:fusing nodes _matching_feat_encoder_layer1_layer1_1_conv2_Conv,_matching_feat_encoder_layer1_layer1

In [3]:
# Get statistics
multiple_input_for_statistics = np.load("./statistics_small.npy")[:5]
stats = G.collect_statistics(multiple_input_for_statistics)

node_options = {}
node_options["output_1"] = {"scheme": "FLOAT", "float_type": "float16"}
node_options["output_2"] = {"scheme": "FLOAT", "float_type": "float16"}
node_options["output_3"] = {"scheme": "FLOAT", "float_type": "float16"}

G.quantize(
    statistics=stats,
    graph_options=quantization_options(
        #weight_bits=8,
        use_ne16=True,
        hwc=True
    ),
    node_options=node_options
)

INFO:nntool.graph.nngraph:update graph dimensions
INFO:nntool.graph.nngraph:update graph dimensions
INFO:nntool.graph.manipulations.adjust_order:adding transposes to correct tensor order for AT kernels
INFO:nntool.graph.manipulations.eliminate_transposes.eliminate_transposes:found elimination for _ctx_feat_encoder_conv1_Conv_fusion_trans_in0 upwards - 3 eliminated
INFO:nntool.graph.manipulations.eliminate_transposes.eliminate_transposes:found elimination for _ctx_feat_encoder_conv1_Conv_fusion_trans_in1 upwards - 1 eliminated
INFO:nntool.graph.manipulations.eliminate_transposes.eliminate_transposes:found elimination for _ctx_feat_encoder_conv1_Conv_fusion_trans_out0 downwards - 2 eliminated
INFO:nntool.graph.manipulations.eliminate_transposes.eliminate_transposes:found elimination for _ctx_feat_encoder_conv2_Conv_fusion_trans_in0 upwards - 2 eliminated
INFO:nntool.graph.manipulations.eliminate_transposes.eliminate_transposes:found elimination for _ctx_feat_encoder_conv2_Conv_fusion_tra

In [4]:
# Get input
input_data = multiple_input_for_statistics[0][np.newaxis, :]
input_data = np.transpose(input_data, (0, 1, 3, 4, 2)) # NHWC --> NCHW

#Quantised execution
quant_execution = G.execute([input_data], quantize=True, dequantize=True)

In [12]:
# Execution on target on GVSOC
input_for_gap9 = G.execute([input_data], dequantize=False, quantize=True)

for input_node in G.input_nodes():
    input_node.allocate = 1

for output_node in G.output_nodes():
    output_node.allocate = 1
    output_node.at_options.out_home_mem_loc = "AT_MEM_L3_DEFAULTRAM"

work_dir = "test_tutorial"
res = G.execute_on_target(
    pmsis_os='freertos',
    platform="gvsoc",
    directory=work_dir,
    input_tensors=input_for_gap9[0],
    write_out_to_file=True,
    settings=model_settings(
        l1_size=108000,
        l2_size=800000, #Change overall L2 allowed here
        l3_size=16000000, #Change overall L3 allowed here
        tensor_directory='./tensors',
        graph_const_exec_from_flash=True,
        graph_checksum=True,
        #graph_dump_tensor=6, # Uncomment this line to dump tensor, cannot run with checksum
        #graph_dump_tensor_to_file=True # Uncomment this line to dump tensor to file, cannot run with checksum
    ),
    at_loglevel=2,
    print_output=True
)

INFO:nntool.graph.nngraph:update graph dimensions


Script started, output log file is '/tmp/tmpq9d74con/log.txt'.
The target board you have sourced is : gap9_evk, GAP9_V2.
Release version: 5.20.4
-- [SDK] Version: 5.20.4
-- [Platform] GVSoC
-- [OS] FreeRTOS (Scheduler v2)
-- [Printf] Semihost
-- [Board] GAP Family : 9
-- [Board] GAP Version : 2
-- [AutoTiler] Setup model: patchifier_rpg
-- Clock speeds - Cluster 370 FC 370 Peripheral 370
-- [Log] none

--- patchifier_rpg options ---

-- [patchifier_rpg] Include directories : /home/dzhong/patchifier_nntool_test/test_tutorial;/home/dzhong/gap_sdk_private/tools/autotiler_v3/Autotiler;/home/dzhong/gap_sdk_private/tools/autotiler_v3/Emulation;/home/dzhong/gap_sdk_private/libs/gap_lib/include;/home/dzhong/gap_sdk_private/utils/power_meas_utils;/home/dzhong/gap_sdk_private/libs/include;/home/dzhong/gap_sdk_private/tools/autotiler_v3/ISP_Libraries;/home/dzhong/gap_sdk_private/tools/autotiler_v3/ISP_Libraries;/home/dzhong/gap_sdk_private/tools/autotiler_v3/CNN_Libraries;/home/dzhong/gap_sdk_pri

In [6]:
# Problem: The attempt to calculate the checksum of S33 resulted in an Invalid access error
# Solution: In AT_ChecksumTensor function, move the IsFloat check before the SizeToRead calculation